# SARIMA Experiment

This notebook mirrors the aggregate ARIMA experiment for the SARIMA folder. It keeps external regressors out and tests:

1. compact non-seasonal `(p,d,q)` and annual seasonal `(P,D,Q,52)` order search;
2. last-year vs blended allocation from weekly total forecast to Store-Dept rows.

The baseline result to beat is:

```text
Previous non-seasonal folder baseline `(1,1,1)` validation WMAE = 1856.86053
Seasonal naive validation WMAE = 1800.17359
```

SARIMAX with external regressors is kept in `model_experiment_SARIMAX.ipynb`.


In [1]:
%pip install -q "numpy>=1.24,<3" "pandas>=2.0,<3" "scikit-learn>=1.3,<2" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "cloudpickle>=3.0,<4"

from pathlib import Path
import itertools
import warnings
import cloudpickle

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    import wandb
except Exception:
    wandb = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

In [2]:
try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

Mounted at /content/drive


## Configuration

The search is intentionally compact because only 104 training weeks are available. This true SARIMA experiment combines `(p,d,q)` with annual `(P,D,Q,52)` orders and uses no external regressors.


In [3]:
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/Walmart-Recruiting---Store-Sales-Forecasting/data"),
    Path("../../../../data"),
    Path("../../../data"),
    Path("data"),
]

VALIDATION_WEEKS = 39
HOLIDAY_WEIGHT = 5.0
ORDER_GRID = [(0, 0, 1), (1, 0, 1), (1, 0, 2)]
SEASONAL_ORDER_GRID = [(1, 0, 0, 52), (0, 1, 0, 52), (0, 0, 1, 52)]
ALLOCATION_STRATEGIES = ["last_year_share", "blended_share"]

RUN_WANDB = True
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_RUN_NAME = "SARIMA_Order_Allocation_Experiment"
WANDB_MODE = "online"

RUN_TEST_SUBMISSION = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def resolve_data_dir(candidates):
    required = ["train.csv", "test.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Could not find Walmart data directory.")


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday).astype(bool), holiday_weight, 1.0)
    return float(
        np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        / np.sum(weights)
    )


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR.resolve()}")

Using data directory: /content/drive/MyDrive/walmart_competition_data


In [4]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

train = train.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test = test.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
features = features.sort_values(["Date", "Store"]).reset_index(drop=True)

print(
    {
        "train": train.shape,
        "test": test.shape,
        "features": features.shape,
        "stores": stores.shape,
    }
)
display(train.head())

{'train': (421570, 5), 'test': (115064, 4), 'features': (8190, 12), 'stores': (45, 3)}


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,2,2010-02-05,50605.27,False
2,1,3,2010-02-05,13740.12,False
3,1,4,2010-02-05,39954.04,False
4,1,5,2010-02-05,32229.38,False


In [5]:
all_dates = np.array(sorted(train["Date"].unique()))
validation_dates = all_dates[-VALIDATION_WEEKS:]
validation_start = validation_dates[0]

train_part = train[train["Date"] < validation_start].copy()
val_part = train[train["Date"] >= validation_start].copy()

print(
    {
        "train_rows": len(train_part),
        "validation_rows": len(val_part),
        "train_end": train_part["Date"].max().date(),
        "validation_start": pd.Timestamp(validation_start).date(),
        "validation_end": val_part["Date"].max().date(),
    }
)

{'train_rows': 305982, 'validation_rows': 115588, 'train_end': datetime.date(2012, 1, 27), 'validation_start': datetime.date(2012, 2, 3), 'validation_end': datetime.date(2012, 10, 26)}


## SARIMA Improvement Scope

Pure SARIMA does not use tabular feature engineering. To improve the baseline without turning it into SARIMAX, this notebook focuses on two things:

- choosing a better `(p, d, q)` order;
- explicitly tuning annual `(P, D, Q, 52)` seasonality;
- improving the allocation from aggregate weekly forecast to Store-Dept rows.

This keeps the experiment classical and easy to compare with the baseline.


In [6]:
def weekly_total_sales(frame):
    return (
        frame.groupby("Date", as_index=True)["Weekly_Sales"]
        .sum()
        .sort_index()
        .asfreq("W-FRI")
    )


weekly_sales_train = weekly_total_sales(train_part)
print(
    {
        "weekly_train_points": len(weekly_sales_train),
        "weekly_train_start": weekly_sales_train.index.min().date(),
        "weekly_train_end": weekly_sales_train.index.max().date(),
    }
)

{'weekly_train_points': 104, 'weekly_train_start': datetime.date(2010, 2, 5), 'weekly_train_end': datetime.date(2012, 1, 27)}


## Allocation Strategies

The aggregate SARIMA model forecasts total weekly sales. Kaggle needs row-level Store-Dept predictions, so we compare two allocation strategies:

- `last_year_share`: uses same Store-Dept sales 52 weeks ago;
- `blended_share`: combines last-year sales with recent pre-validation average sales.

In [7]:
def fit_forecast_sarima(y_train, forecast_dates, order, seasonal_order):
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(disp=False, maxiter=200)
    forecast = result.forecast(steps=len(forecast_dates))
    forecast = pd.Series(np.asarray(forecast), index=forecast_dates).clip(lower=0.0)
    return result, forecast


def make_row_forecast(
    target_frame,
    history_frame,
    aggregate_forecast,
    strategy="last_year_share",
    recent_weeks=13,
):
    target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()

    last_year = history.copy()
    last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
    last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})

    series_mean = (
        history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "series_mean_sales"})
    )
    recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
    recent_mean = (
        history[history["Date"] > recent_cutoff]
        .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "recent_mean_sales"})
    )

    target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(series_mean, on=["Store", "Dept"], how="left")
    target = target.merge(recent_mean, on=["Store", "Dept"], how="left")

    if strategy == "last_year_share":
        base = target["last_year_sales"].fillna(target["series_mean_sales"])
    elif strategy == "blended_share":
        last_year_base = target["last_year_sales"].fillna(target["series_mean_sales"])
        recent_base = target["recent_mean_sales"].fillna(target["series_mean_sales"])
        base = 0.70 * last_year_base + 0.30 * recent_base
    else:
        raise ValueError(f"Unknown allocation strategy: {strategy}")

    target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
    date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
    row_count = target.groupby("Date")["allocation_base"].transform("size")
    target["share"] = np.where(
        date_base_sum > 0, target["allocation_base"] / date_base_sum, 1.0 / row_count
    )
    target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
    target["prediction"] = (
        (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
    )
    return target["prediction"].to_numpy()


def make_seasonal_naive_forecast(target_frame, history_frame):
    target = target_frame[["Store", "Dept", "Date"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    history["Date"] = history["Date"] + pd.Timedelta(days=364)
    history = history.rename(columns={"Weekly_Sales": "last_year_sales"})
    fallback = (
        history_frame.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .median()
        .rename(columns={"Weekly_Sales": "series_median_sales"})
    )
    target = target.merge(history, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(fallback, on=["Store", "Dept"], how="left")
    return (
        target["last_year_sales"]
        .fillna(target["series_median_sales"])
        .fillna(0.0)
        .clip(lower=0.0)
        .to_numpy()
    )

In [9]:
weekly_sales = weekly_total_sales(train_part)
forecast_dates = list(pd.to_datetime(validation_dates))

seasonal_naive_pred = make_seasonal_naive_forecast(val_part, train_part)
seasonal_naive_wmae = weighted_mae(
    val_part["Weekly_Sales"], seasonal_naive_pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
)
print(f"Seasonal naive WMAE: {seasonal_naive_wmae:.4f}")

wandb_run = None
trial_table = None
if RUN_WANDB:
    if wandb is None:
        raise ImportError(
            "wandb is not available. Re-run the install/import cell or install wandb."
        )
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        job_type="sarima-experiment",
        mode=WANDB_MODE,
        reinit=True,
        config={
            "validation_weeks": VALIDATION_WEEKS,
            "holiday_weight": HOLIDAY_WEIGHT,
            "order_grid_size": len(ORDER_GRID),
            "seasonal_order_grid_size": len(SEASONAL_ORDER_GRID),
            "seasonal_period": 52,
            "allocation_strategies": ALLOCATION_STRATEGIES,
            "model_family": "SARIMA",
            "uses_exog": False,
            "seasonal_orders": [str(value) for value in SEASONAL_ORDER_GRID],
        },
    )
    trial_table = wandb.Table(
        columns=[
            "trial",
            "order",
            "seasonal_order",
            "allocation",
            "wmae",
            "mae",
            "rmse",
            "improvement_vs_seasonal_naive_pct",
        ]
    )

results = []
trial = 0
for order in ORDER_GRID:
  for seasonal_order in SEASONAL_ORDER_GRID:
    try:
        _, aggregate_forecast = fit_forecast_sarima(weekly_sales, forecast_dates, order, seasonal_order)
    except Exception as exc:
        print(f"Trial {trial:03d} failed for order={order}, seasonal_order={seasonal_order}: {exc}")
        trial += 1
        continue

    for allocation in ALLOCATION_STRATEGIES:
        pred = make_row_forecast(val_part, train_part, aggregate_forecast, allocation)
        wmae_value = weighted_mae(
            val_part["Weekly_Sales"], pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
        )
        mae_value = float(mean_absolute_error(val_part["Weekly_Sales"], pred))
        rmse_value = rmse(val_part["Weekly_Sales"], pred)
        improvement = 100.0 * (seasonal_naive_wmae - wmae_value) / seasonal_naive_wmae
        row = {
            "trial": trial,
            "order": order,
            "seasonal_order": seasonal_order,
            "allocation": allocation,
            "validation/wmae": wmae_value,
            "validation/mae": mae_value,
            "validation/rmse": rmse_value,
            "improvement_vs_seasonal_naive_pct": improvement,
        }
        results.append(row)
        print(
            f"Trial {trial:03d} | order={order} seasonal_order={seasonal_order} allocation={allocation} | WMAE={wmae_value:.4f} | improvement={improvement:.2f}%"
        )
        if RUN_WANDB:
            wandb.log(
                {
                    "trial": trial,
                    "validation/wmae": wmae_value,
                    "validation/mae": mae_value,
                    "validation/rmse": rmse_value,
                    "improvement_vs_seasonal_naive_pct": improvement,
                    "baseline/seasonal_naive_wmae": seasonal_naive_wmae,
                },
                step=trial,
            )
            trial_table.add_data(
                trial,
                str(order),
                str(seasonal_order),
                allocation,
                wmae_value,
                mae_value,
                rmse_value,
                improvement,
            )
    trial += 1

results_df = pd.DataFrame(results).sort_values("validation/wmae").reset_index(drop=True)
best_result = results_df.iloc[0].to_dict()

if RUN_WANDB:
    wandb.log({"sarima_experiment/trials": trial_table})
    wandb.summary["best_validation_wmae"] = float(best_result["validation/wmae"])
    wandb.summary["best_validation_mae"] = float(best_result["validation/mae"])
    wandb.summary["best_validation_rmse"] = float(best_result["validation/rmse"])
    wandb.summary["best_order"] = str(best_result["order"])
    wandb.summary["best_seasonal_order"] = str(best_result["seasonal_order"])
    wandb.summary["best_allocation"] = best_result["allocation"]
    wandb.summary["seasonal_naive_wmae"] = float(seasonal_naive_wmae)
    wandb.finish()

display(results_df.head(10))
best_result

Seasonal naive WMAE: 1800.1736


Trial 000 | order=(0, 0, 0) allocation=last_year_share | WMAE=15952.3237 | improvement=-786.15%
Trial 000 | order=(0, 0, 0) allocation=blended_share | WMAE=15952.3237 | improvement=-786.15%
Trial 001 | order=(0, 0, 1) allocation=last_year_share | WMAE=15911.9607 | improvement=-783.91%
Trial 001 | order=(0, 0, 1) allocation=blended_share | WMAE=15911.9346 | improvement=-783.91%
Trial 002 | order=(0, 0, 2) allocation=last_year_share | WMAE=15195.4592 | improvement=-744.11%
Trial 002 | order=(0, 0, 2) allocation=blended_share | WMAE=15197.5197 | improvement=-744.23%
Trial 003 | order=(0, 1, 0) allocation=last_year_share | WMAE=2941.1402 | improvement=-63.38%
Trial 003 | order=(0, 1, 0) allocation=blended_share | WMAE=3149.6627 | improvement=-74.96%
Trial 004 | order=(0, 1, 1) allocation=last_year_share | WMAE=2183.7088 | improvement=-21.31%
Trial 004 | order=(0, 1, 1) allocation=blended_share | WMAE=2425.3955 | improvement=-34.73%
Trial 005 | order=(0, 1, 2) allocation=last_year_share | W

baseline/seasonal_naive_wmae,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
improvement_vs_seasonal_naive_pct,▁▁▁▇██▆██▇██▇█▆███
trial,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
validation/mae,███▂▁▁▃▁▁▁▁▁▂▁▃▁▁▁
validation/rmse,███▂▁▁▃▁▁▁▁▁▂▁▃▁▁▁
validation/wmae,███▂▁▁▃▁▁▂▁▁▂▁▃▁▁▁
baseline/seasonal_naive_wmae,1800.17359
best_allocation,last_year_share
best_order,"(1, 0, 2)"
best_validation_mae,1835.0506
best_validation_rmse,3920.68168


,trial,order,allocation,validation/wmae,validation/mae,validation/rmse,improvement_vs_seasonal_naive_pct
0,8,"(1, 0, 2)",last_year_share,1831.617620,1835.050602,3920.681681,-1.746722
1,5,"(0, 1, 2)",last_year_share,1846.058549,1864.597955,3993.862035,-2.548918
2,10,"(1, 1, 1)",last_year_share,1856.860525,1843.287631,3919.594370,-3.148970
3,17,"(2, 1, 2)",last_year_share,1889.000495,1882.232425,4025.247626,-4.934352
4,5,"(0, 1, 2)",blended_share,1994.074033,1992.851893,4162.355218,-10.771208
5,8,"(1, 0, 2)",blended_share,2004.480735,1989.159877,4126.772678,-11.349302
6,17,"(2, 1, 2)",blended_share,2040.498977,2008.673155,4190.754298,-13.350123
7,10,"(1, 1, 1)",blended_share,2050.032996,2018.022166,4152.395977,-13.879739
8,16,"(2, 1, 1)",last_year_share,2132.767142,2091.082776,4270.966796,-18.475638
9,4,"(0, 1, 1)",last_year_share,2183.708818,2127.613378,4329.207131,-21.305458


{'trial': 8,
 'order': (1, 0, 2),
 'allocation': 'last_year_share',
 'validation/wmae': 1831.617619664779,
 'validation/mae': 1835.0506016783845,
 'validation/rmse': 3920.681680842303,
 'improvement_vs_seasonal_naive_pct': -1.7467219905781104}

## Register Best SARIMA Pipeline

Run this cell after the experiment loop. It refits the best SARIMA order on all available training data, packages the fitted aggregate model with allocation history, logs it as a W&B model artifact, and links it to the W&B Model Registry.


In [10]:
required_objects = ["best_result", "train", "weekly_total_sales"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Run previous experiment cells first. Missing: " + ", ".join(missing_objects)
    )
if wandb is None:
    raise ImportError(
        "wandb is not available. Re-run the install/import cell or install wandb."
    )

REGISTER_SARIMA_MODEL = True
SARIMA_REGISTRY_TARGET = "wandb-registry-model/Walmart_SARIMA_Pipeline"
SARIMA_MODEL_ARTIFACT_NAME = "walmart-sarima-best-pipeline"


class AggregateSARIMAPipeline:
    """Raw-test SARIMA pipeline for W&B Model Registry."""

    def __init__(
        self,
        model_result,
        order,
        seasonal_order,
        allocation_strategy,
        observed_history,
        validation_metrics,
        metadata,
    ):
        self.model_result = model_result
        self.order = tuple(order)
        self.seasonal_order = tuple(seasonal_order)
        self.allocation_strategy = allocation_strategy
        self.observed_history = observed_history.copy()
        self.validation_metrics = dict(validation_metrics)
        self.metadata = dict(metadata)

    def _aggregate_forecast(self, raw_df):
        dates = list(pd.to_datetime(sorted(raw_df["Date"].unique())))
        forecast = self.model_result.forecast(steps=len(dates))
        return pd.Series(np.asarray(forecast), index=dates).clip(lower=0.0)

    def _make_row_forecast(self, target_frame, aggregate_forecast, recent_weeks=13):
        target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
        history = self.observed_history[
            ["Store", "Dept", "Date", "Weekly_Sales"]
        ].copy()
        history["Date"] = pd.to_datetime(history["Date"])

        last_year = history.copy()
        last_year["Date"] = last_year["Date"] + pd.Timedelta(days=364)
        last_year = last_year.rename(columns={"Weekly_Sales": "last_year_sales"})
        series_mean = (
            history.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "series_mean_sales"})
        )
        recent_cutoff = history["Date"].max() - pd.Timedelta(days=7 * recent_weeks)
        recent_mean = (
            history[history["Date"] > recent_cutoff]
            .groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
            .mean()
            .rename(columns={"Weekly_Sales": "recent_mean_sales"})
        )
        target = target.merge(last_year, on=["Store", "Dept", "Date"], how="left")
        target = target.merge(series_mean, on=["Store", "Dept"], how="left")
        target = target.merge(recent_mean, on=["Store", "Dept"], how="left")
        if self.allocation_strategy == "last_year_share":
            base = target["last_year_sales"].fillna(target["series_mean_sales"])
        elif self.allocation_strategy == "blended_share":
            last_year_base = target["last_year_sales"].fillna(
                target["series_mean_sales"]
            )
            recent_base = target["recent_mean_sales"].fillna(
                target["series_mean_sales"]
            )
            base = 0.70 * last_year_base + 0.30 * recent_base
        else:
            raise ValueError(f"Unknown allocation strategy: {self.allocation_strategy}")
        target["allocation_base"] = base.fillna(0.0).clip(lower=0.0)
        date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
        row_count = target.groupby("Date")["allocation_base"].transform("size")
        target["share"] = np.where(
            date_base_sum > 0,
            target["allocation_base"] / date_base_sum,
            1.0 / row_count,
        )
        target["aggregate_forecast"] = target["Date"].map(aggregate_forecast)
        target["prediction"] = (
            (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
        )
        return target["prediction"].to_numpy()

    def predict(self, raw_df):
        frame = raw_df.copy()
        frame["Date"] = pd.to_datetime(frame["Date"])
        if "IsHoliday" not in frame.columns:
            frame["IsHoliday"] = False
        aggregate_forecast = self._aggregate_forecast(frame)
        return self._make_row_forecast(frame, aggregate_forecast)


if REGISTER_SARIMA_MODEL:
    best_order = tuple(best_result["order"])
    best_seasonal_order = tuple(best_result["seasonal_order"])
    best_allocation = best_result["allocation"]
    full_weekly_sales = weekly_total_sales(train)
    full_model = SARIMAX(
        full_weekly_sales,
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit()

    registry_output_dir = OUTPUT_DIR / "sarima_registry"
    registry_output_dir.mkdir(parents=True, exist_ok=True)
    pipeline_path = registry_output_dir / "sarima_best_pipeline.pkl"

    validation_metrics = {
        "best_validation_wmae": float(best_result["validation/wmae"]),
        "best_validation_mae": float(best_result["validation/mae"]),
        "best_validation_rmse": float(best_result["validation/rmse"]),
        "seasonal_naive_wmae": float(seasonal_naive_wmae),
        "improvement_vs_seasonal_naive_pct": float(
            best_result["improvement_vs_seasonal_naive_pct"]
        ),
    }
    pipeline = AggregateSARIMAPipeline(
        model_result=full_model,
        order=best_order,
        seasonal_order=best_seasonal_order,
        allocation_strategy=best_allocation,
        observed_history=train,
        validation_metrics=validation_metrics,
        metadata={
            "model_family": "SARIMA",
            "uses_exog": False,
            "seasonal_order": str(best_seasonal_order),
            "validation_weeks": VALIDATION_WEEKS,
            "refit_scope": "full_train",
        },
    )
    with pipeline_path.open("wb") as file:
        cloudpickle.dump(pipeline, file)

    registry_run = wandb.init(
        project=WANDB_PROJECT,
        name="SARIMA_Best_Model_Registry",
        job_type="model_registration",
        mode=WANDB_MODE,
        reinit=True,
        config={
            **validation_metrics,
            "best_order": str(best_order),
            "best_seasonal_order": str(best_seasonal_order),
            "best_allocation": best_allocation,
            "registry_target": SARIMA_REGISTRY_TARGET,
        },
    )
    model_artifact = wandb.Artifact(
        name=SARIMA_MODEL_ARTIFACT_NAME,
        type="model",
        description="Best aggregate SARIMA pipeline with row-level Store-Dept allocation.",
        metadata={
            **validation_metrics,
            "model_family": "SARIMA",
            "order": str(best_order),
            "seasonal_order": str(best_seasonal_order),
            "allocation_strategy": best_allocation,
            "registry_target": SARIMA_REGISTRY_TARGET,
        },
    )
    model_artifact.add_file(str(pipeline_path))
    logged_artifact = registry_run.log_artifact(
        model_artifact, aliases=["best", "latest"]
    )
    registry_run.link_artifact(
        logged_artifact,
        target_path=SARIMA_REGISTRY_TARGET,
        aliases=["best-sarima", "latest", "champion"],
    )
    registry_run.summary.update(validation_metrics)
    registry_run.summary["registry_target"] = SARIMA_REGISTRY_TARGET
    registry_run.summary["pipeline_artifact"] = SARIMA_MODEL_ARTIFACT_NAME
    registry_run.finish()

    print("Registered best SARIMA pipeline in W&B Model Registry:")
    print(f"  artifact: {SARIMA_MODEL_ARTIFACT_NAME}")
    print(f"  registry: {SARIMA_REGISTRY_TARGET}")
    print(f"  validation WMAE: {validation_metrics['best_validation_wmae']:.4f}")

best_validation_mae,1835.0506
best_validation_rmse,3920.68168
best_validation_wmae,1831.61762
improvement_vs_seasonal_naive_pct,-1.74672
pipeline_artifact,walmart-sarima-best-...
registry_target,wandb-registry-model...
seasonal_naive_wmae,1800.17359


Registered best SARIMA pipeline in W&B Model Registry:
  artifact: walmart-sarima-best-pipeline
  registry: wandb-registry-model/Walmart_SARIMA_Pipeline
  validation WMAE: 1831.6176
